# 06 GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

## Inputs

- Screened yearly pairs from notebook 04c.
- Yearly SI and eligible-pair outputs from notebook 05.
- GLASS pair data and the configured pathogen/antibiotic lookup tables.

## Outputs

- Interactive GLASS comparison figures, PDF exports, and supporting CSV tables under the configured `outputs/figures/` directory.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters

In [3]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]
INTERESTING_OUTPUT_PATH = SI_PATHS["interesting_output"]

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Reuse SI fit window settings so interactive plots stay aligned with notebook 05 outputs.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")
MIN_AMR_PAIR_PUBLICATION_COUNT = cfg(PIPELINE_CONFIG, "si.min_amr_pair_publication_count")
MIN_NUM_YEARS_WITH_VALID_COUNTS = cfg(PIPELINE_CONFIG, "si.min_num_years_with_valid_counts")

# Interactive filtering and plot layout controls.
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_TOTAL_PUBLICATIONS = cfg(PIPELINE_CONFIG, "glass_interactive.min_total_publications")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

if not YEARLY_SCREENED_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SCREENED_PATH}. Run notebook 04 to create screened AMR pairs first.")
if not YEARLY_SI_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SI_PATH}. Run notebook 05 after notebook 04 to build SI outputs.")
if YEARLY_SI_PATH.stat().st_mtime < YEARLY_SCREENED_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{YEARLY_SI_PATH} is older than {YEARLY_SCREENED_PATH}. "
        "Run notebook 05 after the latest notebook 04 LLM/screened-pair run."
    )


## Load SI And Clean GLASS List

In [4]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 04",
    modeled_label="filtered SI dataset from notebook 05",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 8,835 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 33; antibiotics/levels: 69
Saved cleaned GLASS rows: 96 to /Users/f.costa/Code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 04 pathogens: 335
filtered SI dataset from notebook 05 pathogens: 33
Pathogens present in screened AMR input from notebook 04 but absent from filtered SI dataset from notebook 05: 302
Achromobacter denitrificans, Achromobacter mucicolens, Achromobacter xylosoxidans, Acinetobacter calcoaceticus, Acinetobacter haemolyticus, Acinetobacter johnsonii, Acinetobacter junii, Acinetobacter lwoffii, Acinetobacter nosocomialis, Acinetobacter pittii, Acinetobacter radioresistens, Acinetobacter ursingii, Actinobacillus equuli, Actinotignum schaalii, Aerococcus viridans, Aeromonas caviae, Aeromonas dhakensis, Aeromonas hydrophila, Aeromonas jandaei, Aeromonas salmonicida, Aeromonas schubertii, Aeromonas sobria, Aeromonas

,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Achromobacter denitrificans,5,5,4,0.0,0.0,0.0,True,False,missing_from_modeled
1,Achromobacter mucicolens,9,9,9,0.0,0.0,0.0,True,False,missing_from_modeled
2,Achromobacter xylosoxidans,116,127,44,0.0,0.0,0.0,True,False,missing_from_modeled
3,Acinetobacter calcoaceticus,66,69,31,0.0,0.0,0.0,True,False,missing_from_modeled
4,Acinetobacter haemolyticus,2,2,2,0.0,0.0,0.0,True,False,missing_from_modeled
...,...,...,...,...,...,...,...,...,...,...
297,Weissella confusa,6,6,5,0.0,0.0,0.0,True,False,missing_from_modeled
298,Winkia neuii,7,7,7,0.0,0.0,0.0,True,False,missing_from_modeled
299,Yersinia enterocolitica,228,341,42,0.0,0.0,0.0,True,False,missing_from_modeled
300,Yersinia pestis,17,21,5,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [5]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    min_num_years_with_valid_counts=MIN_NUM_YEARS_WITH_VALID_COUNTS,
    verbose=True,
)

if not INTERESTING_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {INTERESTING_OUTPUT_PATH}. Run notebook 05 first so only pairs satisfying all SI criteria enter the GLASS analysis."
    )
if INTERESTING_OUTPUT_PATH.stat().st_mtime < YEARLY_SI_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{INTERESTING_OUTPUT_PATH} is older than {YEARLY_SI_PATH}. Rerun notebook 05 so its configured SI eligibility criteria are applied."
    )
eligible_pairs = pd.read_parquet(INTERESTING_OUTPUT_PATH)[["pathogen", "antibiotic"]].drop_duplicates()
rank_df = rank_df.merge(eligible_pairs, on=["pathogen", "antibiotic"], how="inner", validate="one_to_one")
print(f"Eligible SI pairs entering Notebook 06: {len(rank_df):,}")
display(rank_df)

Loaded cached rank rows: 377 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future.parquet
Eligible SI pairs entering Notebook 06: 107


,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,raw_latest_available_year,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Streptococcus agalactiae,Tetracycline,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,76.0,24.0,52.0,52.0
1,Staphylococcus aureus,Ceftriaxone,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,244.0,200.0,44.0,44.0
2,Escherichia coli,Erythromycin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,192.0,156.0,36.0,36.0
3,Klebsiella pneumoniae,Amoxicillin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,156.0,126.0,30.0,30.0
4,Klebsiella pneumoniae,Tetracycline,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,171.0,141.0,30.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,Escherichia coli,Ciprofloxacin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,2.0,2.0,0.0,0.0
103,Escherichia coli,Cefotaxime,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,3.0,3.0,0.0,0.0
104,Staphylococcus aureus,Erythromycin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,15.0,15.0,0.0,0.0
105,Klebsiella pneumoniae,Colistin,2025,2028,3,2000,2023,2025,2025,2026.0,...,NaN,0.6,1,NaN,ok,NaN,29.0,29.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [ ]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_total_publications=MIN_TOTAL_PUBLICATIONS,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

## Display GLASS Matches In Plot

In [ ]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2025.0,2028,223.0,2066.0,291.892481,0.645650,0.720519,0.074870,9.604673,antibiotic
1,Escherichia coli,Colistin,2025.0,2028,144.0,1084.0,171.221155,0.379875,0.423195,0.043320,50.182629,antibiotic
2,Escherichia coli,Meropenem,2025.0,2028,118.0,753.0,222.283128,0.307038,0.461254,0.154215,29.220311,antibiotic
3,Klebsiella pneumoniae,Meropenem,2025.0,2028,99.0,638.0,167.729491,0.352542,0.440547,0.088004,15.567749,antibiotic
4,Escherichia coli,Cefotaxime,2025.0,2028,99.0,968.0,151.783181,0.624182,0.710267,0.086085,65.331337,antibiotic
5,Escherichia coli,Sulfamethoxazole/trimethoprim,2025.0,2028,99.0,1294.0,156.504427,0.514665,0.558849,0.044185,19.643576,antibiotic
6,Klebsiella pneumoniae,Colistin,2025.0,2028,98.0,659.0,149.070357,0.386143,0.478235,0.092092,14.267645,antibiotic
7,Escherichia coli,Ceftriaxone,2025.0,2028,98.0,771.0,134.775798,0.444522,0.535785,0.091263,20.104350,antibiotic
8,Pseudomonas aeruginosa,Meropenem,2025.0,2028,81.0,668.0,126.371445,0.281829,0.342310,0.060481,4.207472,antibiotic
9,Pseudomonas aeruginosa,Ceftazidime,2025.0,2028,79.0,732.0,112.233660,0.391343,0.435255,0.043912,16.940975,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [ ]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 72


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Streptococcus agalactiae,Tetracycline,2025.0,2028,22.0,137.0,41.918777,0.229668,0.491769,0.262100,35.879921
1,Staphylococcus aureus,Daptomycin,2025.0,2028,23.0,207.0,35.810522,0.208006,0.363044,0.155038,30.914491
2,Staphylococcus aureus,Ceftriaxone,2025.0,2028,15.0,108.0,23.515300,-0.221472,-0.072083,0.149389,10.839111
3,Escherichia coli,Tigecycline,2025.0,2028,47.0,293.0,66.502950,0.233768,0.377344,0.143577,14.041147
4,Acinetobacter baumannii,Sulbactam,2025.0,2028,21.0,186.0,44.208627,0.342960,0.484635,0.141675,2.587022
...,...,...,...,...,...,...,...,...,...,...,...
67,Streptococcus pneumoniae,Sulfamethoxazole,2025.0,2028,21.0,288.0,17.746160,-0.016212,-0.001716,0.014496,24.711298
68,Pseudomonas aeruginosa,Ceftolozane/tazobactam,2025.0,2028,26.0,213.0,25.751023,0.386682,0.400748,0.014065,46.296751
69,Streptococcus pneumoniae,Amoxicillin,2025.0,2028,22.0,244.0,14.249990,-0.018356,-0.006585,0.011772,35.488005
70,Helicobacter pylori,Metronidazole,2025.0,2028,51.0,745.0,63.105041,0.192237,0.202230,0.009994,17.030466


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [ ]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_si_shift.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_vs_si_shift.pdf


In [ ]:
latest_count_shift_df.to_csv(
    FIGURE_OUTPUT_DIR / "latest_count_shift_df.csv",
    index=False,
)